# S2: Configuração e Teste de Modelos de Inpainting

Este notebook demonstra o uso dos modelos de inpainting implementados.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

from canon.T4 import list_available_models, get_model
from canon.T4.utils import load_image_and_masks

## Listar Modelos Disponíveis

In [ ]:
available_models = list_available_models()
print("Modelos disponíveis:")
for model_name in available_models:
    print(f"  - {model_name}")

## Carregar Imagem e Máscara de Teste

In [ ]:
image_name = "exemplo"
image, masks = load_image_and_masks(image_name)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
mask_idx = 0

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image_rgb)
axes[0].set_title("Imagem Original")
axes[0].axis('off')

axes[1].imshow(masks[mask_idx], cmap='gray')
axes[1].set_title(f"Máscara {mask_idx+1}")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Testar um Modelo Individual

In [ ]:
model_name = "stable_diffusion"

print(f"Carregando modelo: {model_name}")
model = get_model(model_name, device="cuda")

print("Executando inpainting...")
result = model.inpaint(image_rgb, masks[mask_idx])

print(f"Tempo de inferência: {result['inference_time']:.2f}s")
print(f"Configurações: {result['config']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_rgb)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(masks[mask_idx], cmap='gray')
axes[1].set_title("Máscara")
axes[1].axis('off')

axes[2].imshow(result['image'])
axes[2].set_title(f"Resultado - {model_name}")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Comparar Múltiplos Modelos

In [ ]:
models_to_test = ["stable_diffusion", "kandinsky"]

results = {}
for model_name in models_to_test:
    try:
        print(f"Testando {model_name}...")
        model = get_model(model_name, device="cuda")
        result = model.inpaint(image_rgb, masks[mask_idx])
        results[model_name] = result
        model.unload_model()
        print(f"  Tempo: {result['inference_time']:.2f}s")
    except Exception as e:
        print(f"  Erro: {e}")
        results[model_name] = None

In [ ]:
valid_results = {k: v for k, v in results.items() if v is not None}
num_models = len(valid_results)

fig, axes = plt.subplots(1, num_models + 2, figsize=(6 * (num_models + 2), 6))

axes[0].imshow(image_rgb)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(masks[mask_idx], cmap='gray')
axes[1].set_title("Máscara")
axes[1].axis('off')

for idx, (model_name, result) in enumerate(valid_results.items(), start=2):
    axes[idx].imshow(result['image'])
    axes[idx].set_title(f"{model_name}\n{result['inference_time']:.2f}s")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## Teste com Diferentes Máscaras

In [ ]:
model_name = "stable_diffusion"
model = get_model(model_name, device="cuda")

num_masks = len(masks)
fig, axes = plt.subplots(2, num_masks, figsize=(6 * num_masks, 12))

for idx, mask in enumerate(masks):
    result = model.inpaint(image_rgb, mask)
    
    axes[0, idx].imshow(mask, cmap='gray')
    axes[0, idx].set_title(f"Máscara {idx+1}")
    axes[0, idx].axis('off')
    
    axes[1, idx].imshow(result['image'])
    axes[1, idx].set_title(f"Resultado {idx+1}\n{result['inference_time']:.2f}s")
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

model.unload_model()

In [ ]:
model_name = "kandinsky"

print(f"Carregando modelo: {model_name}")
print(f"Tamanho original da imagem: {image_rgb.shape}")

model = get_model(model_name, device="cuda")

mask = masks[0]

print("Executando inpainting com Kandinsky...")
result = model.inpaint(image_rgb, mask)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_rgb)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title("Máscara")
axes[1].axis('off')

axes[2].imshow(result['image'])
axes[2].set_title(f"Resultado - Kandinsky\n{result['inference_time']:.2f}s")
axes[2].axis('off')

plt.tight_layout()
plt.show()

model.unload_model()

print(f"\n✓ Teste concluído!")

## Próximos Passos

Para restauração de fotos antigas, experimente usar **Paint-by-Example** com uma foto exemplo já restaurada!

## Modelos Disponíveis

Temos 3 modelos de inpainting implementados:

1. **Stable Diffusion Inpainting** 
   - Modelo versátil e rápido
   - Usa prompts de texto (opcional)
   
2. **Kandinsky 2.2 Inpainting**
   - Prompts otimizados para fotos antigas
   - Excelente para restauração vintage
   
3. **Paint-by-Example**
   - Guiado por exemplo visual (não usa texto)
   - Ideal quando você tem uma foto similar já restaurada